In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

In [5]:
file_path_idi = r"C:\Users\lenovo\Desktop\GSARP\naari-framework\data\raw\IDI2026dataset.xlsx"
df_idi = pd.read_excel(file_path_idi, sheet_name='IDI 2026 Data', header=None)
data_rows_idi = df_idi.iloc[4:].copy()

In [6]:
col_map_idi = {
    0: 'Iso3', 1: 'Region', 2: 'Income_Group', 3: 'Economy',
    4: 'Internet_Users_Pct', 6: 'Households_Internet_Pct', 8: 'Mobile_BB_Subs_100',
    10: 'Coverage_3G_Pct', 12: 'Coverage_4G_Pct', 14: 'Mobile_Traffic_GB',
    16: 'Fixed_Traffic_GB', 18: 'Mobile_Basket_GNI', 20: 'Fixed_Basket_GNI',
    22: 'Mobile_Phone_Owners_Pct', 35: 'Universal_Pillar', 36: 'Meaningful_Pillar', 37: 'IDI_Score'
}

In [7]:
df_idi_clean = pd.DataFrame()
for k, v in col_map_idi.items():
    df_idi_clean[v] = data_rows_idi[k]

In [8]:
num_cols_idi = [
    'Internet_Users_Pct', 'Households_Internet_Pct', 'Mobile_BB_Subs_100', 
    'Coverage_3G_Pct', 'Coverage_4G_Pct', 'Mobile_Traffic_GB', 'Fixed_Traffic_GB', 
    'Mobile_Basket_GNI', 'Fixed_Basket_GNI', 'Universal_Pillar', 'Meaningful_Pillar', 'IDI_Score'
]
for col in num_cols_idi:
    df_idi_clean[col] = pd.to_numeric(df_idi_clean[col].astype(str).str.replace('†', '').str.replace('*', ''), errors='coerce')

target_economies = [
    'Nepal (Republic of)', 'Bangladesh', 'Sri Lanka', 'Pakistan', 'Bhutan',
    'China', 'Estonia', 'Singapore', 'United States', 'Rwanda', 'India'
]

In [9]:
df_idi_sub = df_idi_clean[df_idi_clean['Economy'].isin(target_economies)].copy()

# Save baseline to data/processed
baseline_output = '../data/processed/itu_idi_connectivity_baseline.csv'
df_idi_sub.to_csv(baseline_output, index=False)
print(f"Baseline saved successfully to: {baseline_output}")

# Clean country names for visualizations
name_map = {'Nepal (Republic of)': 'Nepal', 'United States': 'United States of America'}
df_idi_sub['Country_Clean'] = df_idi_sub['Economy'].replace(name_map)

Baseline saved successfully to: ../data/processed/itu_idi_connectivity_baseline.csv


### BroadBand Affordability Barriers

In [11]:
plt.figure(figsize=(10, 6))
affordability_df = df_idi_sub.sort_values(by='Fixed_Basket_GNI', ascending=True)

ax1 = sns.barplot(
    data=affordability_df,
    x='Fixed_Basket_GNI',
    y='Country_Clean',
    palette='viridis'
)
plt.title('Infrastructure Bottleneck: Fixed Broadband Basket Price (% GNI per capita)', fontsize=13, weight='bold', pad=15)
plt.xlabel('Fixed Broadband Basket Price (% of GNI per capita)', fontsize=11)
plt.ylabel('')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

for container in ax1.containers:
    ax1.bar_label(container, fmt='%.1f%%', padding=5)

plt.tight_layout()
plt.savefig("../outputs/figures/connectivity_fixed_broadband_affordability.png", dpi=300, bbox_inches='tight')
plt.close()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_23204\998343084.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax1 = sns.barplot(


In [12]:
plt.figure(figsize=(10, 6))
melted = df_idi_sub.melt(id_vars=['Country_Clean'], value_vars=['Universal_Pillar', 'Meaningful_Pillar'], var_name='Pillar', value_name='Score')
melted['Pillar'] = melted['Pillar'].replace({'Universal_Pillar': 'Universal Connectivity', 'Meaningful_Pillar': 'Meaningful Connectivity'})

ax2 = sns.barplot(
    data=melted,
    x='Score',
    y='Country_Clean',
    hue='Pillar',
    palette='Set2'
)
plt.title('ITU IDI Pillars: Universal vs. Meaningful Connectivity Performance', fontsize=13, weight='bold', pad=15)
plt.xlabel('Normalized Score (0-100)', fontsize=11)
plt.ylabel('')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("../outputs/figures/connectivity_pillars_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

print("Figures successfully generated and saved to outputs/figures/")

Figures successfully generated and saved to outputs/figures/


In [14]:
# 2. Load the baseline datasets
df_itu = pd.read_csv('../data/processed/itu_idi_connectivity_baseline.csv')
df_ox = pd.read_csv('../data/processed/naari_subdimensions_results.csv')

In [15]:
if 'Country_Clean' not in df_itu.columns:
    df_itu['Country_Clean'] = df_itu['Economy'].replace(name_map)

In [16]:
df_merged = pd.merge(df_itu, df_ox, left_on='Country_Clean', right_on='Country', how='inner')

In [17]:
sustainability_metrics = {
    'DPI & Interoperability': ['Data quality', 'e-Government delivery'],
    'Affordability': ['Fixed_Basket_GNI', 'Mobile_Basket_GNI'],
    'Energy & Compute': ['Compute capacity', 'Enabling technical infrastructure'],
    'Talent Retention': ['Human capital'],
    'Cybersecurity': ['Safety and security', 'Governance']
}

In [18]:
target_cols = [col for sublist in sustainability_metrics.values() for col in sublist]
available_cols = [c for c in target_cols if c in df_merged.columns]

In [19]:
df_analysis = df_merged[available_cols].apply(pd.to_numeric, errors='coerce').dropna()
corr_matrix = df_analysis.corr()

In [20]:
# Export matrix
corr_matrix.to_csv('../data/processed/sustainability_correlation_matrix.csv')
print("Saved correlation matrix to ../data/processed/sustainability_correlation_matrix.csv")

Saved correlation matrix to ../data/processed/sustainability_correlation_matrix.csv


In [22]:
p_value_matrix = pd.DataFrame(np.zeros_like(corr_matrix), columns=corr_matrix.columns, index=corr_matrix.index)
for col1 in df_analysis.columns:
    for col2 in df_analysis.columns:
        stat, p = pearsonr(df_analysis[col1], df_analysis[col2])
        p_value_matrix.loc[col1, col2] = p

p_value_matrix.to_csv('../data/processed/sustainability_pvalues_matrix.csv')

In [24]:
plt.figure(figsize=(12, 10))
sns.set_theme(style="white")

# Create a custom mask to hide the upper triangle for a cleaner look
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

ax = sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    vmin=-1, vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson r (Correlation)'}
)

plt.title('Structural Sustainability: Cross-Index Correlation of NAARI Variables', fontsize=14, weight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()

plt.savefig("../outputs/figures/sustainability_correlation_heatmap.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved visualization to ../outputs/figures/sustainability_correlation_heatmap.png")

Saved visualization to ../outputs/figures/sustainability_correlation_heatmap.png
